# generate yaml files for scaling law 


# use hardware constraint to impose the constraint on this scaling law testing.

to prevent from hitting out of memory error:

constraint on:

- model size 
- sequence length 
- microbatch size 
- optimizer setup
- precision 
- activation checkpointing 



hardware limit = f(model size , seq length, microbatch , optimizer/training setup)


runtime budget =  f( model size , data/token count)



# config.yaml

hyperparameters:

  block_size: 64

  batch_size: 64

  n_layer: 6

  n_head: 6

  d_model: 384

  d_compression: 128

  dropout: 0.1

  nb_features: 256

  lr: 0.003         # Scientific notation (6e-3) is also supported

  weight_decay: 0.05

  evals_per_epoch: 3

  attn_type: "flash"

  epochs: 7

  seed: 1337


  log_file: "./logs/mainrun.log"

  data_dir: "./artifacts/datasets/hn_1M"

  output_dir: "./artifacts/runs/train/run_2026_03_15_175902"

  accumulation_steps: 4


In [6]:
hyperparams_dict = {
    'block_size': 64,
    'batch_size': 64,
    'n_layer': 6,
    'n_head': 6,
    'd_model': 384,
    'd_compression': 128,
    'dropout': 0.1,
    'nb_features': 256,
    'lr': 0.003,         # Scientific notation (6e-3) is also supported
    'weight_decay': 0.05,
    'evals_per_epoch': 3,
    'attn_type': "standard",
    'epochs': 7,
    'seed': 1337,
    'log_file': "./logs/mainrun.log",
    'data_dir': "./artifacts/datasets/hn_1M",
    'output_dir': "./artifacts/runs/train/run_2026_03_15_175902",
    'accumulation_steps': 4
}

In [5]:
model_list_from_paper = [
    {
        'weight_decay': 0.0001, # divided by learnign rate 
        'block_size': 2048, 
        'n_layer': 8,  'n_heads': 4, 'd_model': 96, "warmup": 100, 'lr': 0.003, 'batch_size': 64},
    {'weight_decay': 0.0001, 'block_size': 2048, 'n_layer': 8, ' n_heads': 4, 'd_model': 512, "warmup": 400, 'lr': 0.003, 'batch_size': 512},
    {'weight_decay': 0.0001, 'block_size': 2048, 'n_layer': 24, 'n_heads': 8, 'd_model': 576, "warmup": 400, 'lr': 0.003, 'batch_size': 512},
    {'weight_decay': 0.0001, 'block_size': 2048, 'n_layer': 24, 'n_heads': 8, 'd_model': 1024, "warmup": 2000, 'lr': 0.003, 'batch_size': 512},
    {'weight_decay': 0.0001, 'block_size': 2048, 'n_layer': 24, 'n_heads': 16, 'd_model': 2048, "warmup": 5000, 'lr': 0.003, 'batch_size': 256},
    {'weight_decay': 0.0001, 'block_size': 2048, 'n_layer': 32, 'n_heads': 32, 'd_model': 4086, "warmup": 5000, 'lr': 0.0003, 'batch_size': 2048},
] # from the over training paper

# parameter
model_list = [
    {   
        'weight_decay': 0.0001, # divided by learnign rate 
        'block_size': 2048, 
        'n_layer': 3, 
        'd_model': 96,
        'warmup':
    },
    {'n_layer': 4, 'd_model': 128},
    {'n_layer': 5, 'd_model': 160},
    {'n_layer': 6, 'd_model': 224},
    {'n_layer': 8, 'd_model': 288},
    {'n_layer': 9, 'd_model': 320},
    {'n_layer': 10, 'd_model': 384},
    {'n_layer': 12, 'd_model': 480},
    {'n_layer': 14, 'd_model': 576},
    {'n_layer': 15, 'd_model': 640},
    {'n_layer': 18, 'd_model': 704},
    {'n_layer': 21, 'd_model': 832},
    {'n_layer': 23, 'd_model': 1024},
    {'n_layer': 26, 'd_model': 1120},
    {'n_layer': 26, 'd_model': 1312},
    {'n_layer': 30, 'd_model': 1504}
]   # from the discrepancy 

# swiglu (done)
# adamW (done)
# warm up  (done)
# roational positionla embedding  ( doign )s
# implemetned bfloat16 automatic mixed precision
# depthg scaled initialized (done)




# use xFormers see how to 




ModuleNotFoundError: No module named 'triton'

In [ ]:

from omegaconf import OmegaConf
from model.config import Hyperparameters
h = {
    'batch_size': 64,
    'lr': 0.003,
    'weight_decay': 0.05,
    'evals_per_epoch': 3,
    
    'attn_type': "standard",
    'accumulation_steps': 4,
    'warmup_step': 3000,
    
    'n_layer': 8,
    'n_head': 4,
    'd_model': 288,
    'block_size': 2048,
    'dropout': 0.1,
    'is_RoPE': True,

    'vocab_size': 50432,
    
    'mlp_type': 'swiglu'


}

schema = OmegaConf.structured(Hyperparameters)
cfg = OmegaConf.merge(schema, OmegaConf.create(h))

In [6]:
from model.config import GPTConfig
from model.transformer import GPT

model_cfg = GPTConfig.from_flat(cfg)
model = GPT(model_cfg)

model_params = int(sum(p.numel() for p in model.parameters()))
trainable_model_params = int(
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

print("Total params:", model_params)
print("Trainable params:", trainable_model_params)

Total params: 22506048
Trainable params: 22506048


In [11]:
def format_param_count(n: int) -> str:
    if n >= 1_000_000_000:
        return f"{n // 1_000_000_000}B"
    elif n >= 1_000_000:
        return f"{n // 1_000_000}M"
    elif n >= 1_000:
        return f"{n // 1_000}K"
    else:
        return str(n)

In [12]:
print("Total params:", format_param_count(model_params))
print("Trainable params:", format_param_count(trainable_model_params))


Total params: 22M
Trainable params: 22M


In [14]:
from pathlib import Path
import math
from omegaconf import OmegaConf

# --------------------------------------------------
# model shapes you want configs for
# --------------------------------------------------
shapes = [
    # {"depth": 3, "width": 96,  'n_head': 4},
    # {"depth": 4, "width": 128, 'n_head': 4} ,
    # {"depth": 5, "width": 160, 'n_head': 4},
    # {"depth": 6, "width": 224, 'n_head': 4},
    # {"depth": 8, "width": 288, 'n_head': 4},
    {'depth': 9 , "width":  320, }
]

# --------------------------------------------------
# dataset / tokenizer config
# --------------------------------------------------
DATASET_NAME = "tiiuae/falcon-refinedweb"
DATASET_SPLIT = "train"
TEXT_FIELD = "content"
SEED = 1337

VOCAB_SIZE = 50432
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"
REUSE_EXISTING = False
VOCAB_FILENAME = "vocab.json"

# keep val fixed across runs
TARGET_VAL_TOKENS = 5_000_000

# where to save yaml files
CONFIG_OUT_DIR = Path("configs/generated_refinedweb")
CONFIG_OUT_DIR.mkdir(parents=True, exist_ok=True)


def openlm_d_ff(width: int) -> int:
    # rounded SwiGLU hidden size used in the resolving-discrepancies setup
    return 256 * math.ceil(int(8 * width / 3) / 256)


def n_exact(width: int, depth: int, vocab_size: int = VOCAB_SIZE) -> int:
    # N_exact = (4d + 3d_ff) * d * l + d * v
    d_ff = openlm_d_ff(width)
    return ((4 * width + 3 * d_ff) * width * depth) + (width * vocab_size)


for shape in shapes:
    depth = shape["depth"]
    width = shape["width"]

    N_exact = n_exact(width, depth)
    target_train_tokens = 20 * N_exact

    run_name = f"refinedweb_Nexact{format_param_count(N_exact)}"
    output_dir = f"artifacts/datasets/{run_name}"

    cfg = {
        "dataset": {
            "source_type": "huggingface",
            "dataset_name": DATASET_NAME,
            "dataset_split": DATASET_SPLIT,
            "text_field": TEXT_FIELD,
            "output_dir": output_dir,
            "seed": SEED,
            "target_train_tokens": int(target_train_tokens),
            "target_val_tokens": int(TARGET_VAL_TOKENS),
        },
        "tokenizer": {
            "vocab_size": VOCAB_SIZE,
            "eos_token": EOS_TOKEN,
            "unk_token": UNK_TOKEN,
            "reuse_existing": REUSE_EXISTING,
            "vocab_filename": VOCAB_FILENAME,
        },
    }

    save_path = CONFIG_OUT_DIR / f"{run_name}.yaml"
    OmegaConf.save(config=OmegaConf.create(cfg), f=str(save_path))

    print(
        f"saved: {save_path} | "
        f"N_exact={N_exact:,} | "
        f"target_train_tokens={target_train_tokens:,} | "
        f"target_val_tokens={TARGET_VAL_TOKENS:,}"
    )

saved: configs\generated_refinedweb\refinedweb_Nexact5M.yaml | N_exact=5,173,248 | target_train_tokens=103,464,960 | target_val_tokens=5,000,000
saved: configs\generated_refinedweb\refinedweb_Nexact7M.yaml | N_exact=7,503,872 | target_train_tokens=150,077,440 | target_val_tokens=5,000,000
saved: configs\generated_refinedweb\refinedweb_Nexact9M.yaml | N_exact=9,809,920 | target_train_tokens=196,198,400 | target_val_tokens=5,000,000
saved: configs\generated_refinedweb\refinedweb_Nexact15M.yaml | N_exact=15,597,568 | target_train_tokens=311,951,360 | target_val_tokens=5,000,000
saved: configs\generated_refinedweb\refinedweb_Nexact22M.yaml | N_exact=22,487,040 | target_train_tokens=449,740,800 | target_val_tokens=5,000,000


In [15]:
h = {
    'seed': 1337,
    'log_file': "./logs/mainrun.log",
    'data_dir': "./artifacts/datasets/hn_1M",
    'output_dir': "./artifacts/runs/train",

    'lr': 3e-3,         # Scientific notation (6e-3) is also supported
    'weight_decay': 1e-4,
    'evals_per_epoch': 3,
    'betas': [0.9, 0.95],
    'batch_size': 256,          # batch per processing
    'accumulation_steps': 1,   # num_minibatch = batch * accumulation_steps
    'warmup_step': 3000,
    'z_loss_weight': 1e-4,

    'vocab_size': 50432,
    
    'attn_type': "standard",
    'block_size': 2048,
    'n_layer': 8,
    'n_head': 4,
    'd_model': 288,
    'dropout': 0.1,
    'is_RoPE': True,
    'mlp_type': 'standard'


}

schema = OmegaConf.structured(Hyperparameters)
cfg = OmegaConf.merge(schema, OmegaConf.create(h))
from model.config import GPTConfig
from model.transformer import GPT

model_cfg = GPTConfig.from_flat(cfg)
model = GPT(model_cfg)

model_params = int(sum(p.numel() for p in model.parameters()))
trainable_model_params = int(
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

print("Total params:", model_params)
print("Trainable params:", trainable_model_params)

Total params: 22517568
Trainable params: 22517568
